In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Avvio Modulo 01: Data Quality & Integrity Check...\n")

# --- 1. CONFIGURAZIONE PERCORSI (CROSS-PLATFORM) ---
# Otteniamo la directory corrente in cui stiamo eseguendo lo script/notebook
current_dir = Path.cwd()

# Se stiamo eseguendo il notebook da dentro la cartella 'notebooks', 
# dobbiamo "salire" di un livello per trovare 'data'
if current_dir.name == 'notebooks':
    BASE_DIR = current_dir.parent
else:
    BASE_DIR = current_dir

# Costruiamo il percorso dinamico verso i file
DATA_DIR = BASE_DIR / "data" / "raw"

file_fut_03 = DATA_DIR / "ES 03-26.Last.txt"
file_fut_06 = DATA_DIR / "ES 06-26.Last.txt"
file_cfd = DATA_DIR / "fpmarkets_us500_ticks.csv"

print(f"Percorso Dati rilevato: {DATA_DIR}\n")

# --- 2. FUNZIONE DI SCANSIONE ULTRA-LEGGERA FUTURE ---
def analizza_integrita_future(file_path):
    # Usiamo file_path.name per stampare solo il nome del file e non tutto il percorso
    print(f"Scansione integrità Future: {file_path.name}")
    
    # Leggiamo solo la colonna 0 (Timestamp) per non intaccare la memoria
    df = pd.read_csv(file_path, sep=';', usecols=[0], names=['RawTime'], engine='c')
    
    # Pulizia e conversione in formato data
    df['CleanTime'] = df['RawTime'].str.slice(0, 15)
    df['Datetime'] = pd.to_datetime(df['CleanTime'], format='%Y%m%d %H%M%S')
    
    # Calcolo della derivata temporale: quanto tempo passa tra una riga e la successiva?
    df['Time_Gap'] = df['Datetime'].diff()
    
    # Isoliamo i "buchi" anomali. 
    soglia_allarme = pd.Timedelta(hours=3)
    gaps_anomali = df[df['Time_Gap'] > soglia_allarme]
    
    print(f"  -> Righe rilevate: {len(df):,}")
    print(f"  -> Primo Tick: {df['Datetime'].min()}")
    print(f"  -> Ultimo Tick: {df['Datetime'].max()}")
    print(f"  -> Pause di contrattazione o Weekend rilevati: {len(gaps_anomali)}")
    print("-" * 50)
    
    # Liberiamo la RAM forzatamente
    del df
    return gaps_anomali

# --- 3. FUNZIONE DI SCANSIONE CFD (CON FIX ISO8601) ---
def analizza_integrita_cfd(file_path):
    print(f"Scansione integrità CFD: {file_path.name}")
    
    # Leggiamo solo la colonna del tempo
    df = pd.read_csv(file_path, sep=';', usecols=['datetime_utc'], engine='c')
    
    # FIX: format='ISO8601' gestisce i timestamp che troncano i millisecondi (es. i secondi netti)
    df['Datetime'] = pd.to_datetime(df['datetime_utc'], format='ISO8601')
    
    print(f"  -> Righe rilevate: {len(df):,}")
    print(f"  -> Primo Tick: {df['Datetime'].min()}")
    print(f"  -> Ultimo Tick: {df['Datetime'].max()}")
    print("-" * 50)
    
    del df

# --- 4. ESECUZIONE DIAGNOSTICA CON CONTROLLO DI SICUREZZA ---
# Prima di eseguire, controlliamo che i file esistano davvero sul Mac
if not file_fut_03.exists():
    print(f"ERRORE CRITICO: Non trovo il file {file_fut_03.name} nella cartella {DATA_DIR}")
elif not file_fut_06.exists():
    print(f"ERRORE CRITICO: Non trovo il file {file_fut_06.name} nella cartella {DATA_DIR}")
elif not file_cfd.exists():
    print(f"ERRORE CRITICO: Non trovo il file {file_cfd.name} nella cartella {DATA_DIR}")
else:
    # Se i file ci sono tutti, lanciamo le scansioni
    gaps_03 = analizza_integrita_future(file_fut_03)
    gaps_06 = analizza_integrita_future(file_fut_06)
    analizza_integrita_cfd(file_cfd)

    print("\nDIAGNOSTICA COMPLETATA. Dati pronti per il modulo 02 (Merge Vettoriale).")

Avvio Modulo 01: Data Quality & Integrity Check...

Percorso Dati rilevato: /Users/leonardosposato/Documents/git/QuantEdge_Project/data/raw

Scansione integrità Future: ES 03-26.Last.txt
  -> Righe rilevate: 58,024,732
  -> Primo Tick: 2025-12-28 16:38:09
  -> Ultimo Tick: 2026-03-12 22:59:55
  -> Pause di contrattazione o Weekend rilevati: 20
--------------------------------------------------
Scansione integrità Future: ES 06-26.Last.txt
  -> Righe rilevate: 69,184,613
  -> Primo Tick: 2026-03-12 23:00:00
  -> Ultimo Tick: 2026-06-10 14:44:09
  -> Pause di contrattazione o Weekend rilevati: 14
--------------------------------------------------
Scansione integrità CFD: fpmarkets_us500_ticks.csv
  -> Righe rilevate: 25,074,997
  -> Primo Tick: 2026-02-09 01:00:01.679000+00:00
  -> Ultimo Tick: 2026-06-09 23:59:59.546000+00:00
--------------------------------------------------

DIAGNOSTICA COMPLETATA. Dati pronti per il modulo 02 (Merge Vettoriale).
